## Chapter - 1

In [1]:
# pip install pika

In [2]:
# python producer.py

import pika

connection = pika.BlockingConnection(
    pika.ConnectionParameters('localhost')
)

channel = connection.channel()

channel.queue_declare(
    queue='hello',
    durable=True
)
channel.basic_publish(
    exchange='',
    routing_key='hello',
    body='Hello RabbitMQ',
    properties=pika.BasicProperties(
        delivery_mode=2
    )
)

print("Message Sent")

connection.close()

Message Sent


In [3]:
# python consumer.py

# import pika

# connection = pika.BlockingConnection(
#     pika.ConnectionParameters('localhost')
# )

# channel = connection.channel()

# channel.queue_declare(
#     queue='hello',
#     durable=True
# )

# def callback(ch, method, properties, body):
#     print("Received:", body.decode())

# channel.basic_consume(
#     queue='hello',
#     on_message_callback=callback,
#     auto_ack=True
# )

# print("Waiting for messages...")

# channel.start_consuming()

In [4]:
# consumer.py (alternative way to consume a single message) good for testing in notebook

connection = pika.BlockingConnection(
    pika.ConnectionParameters('localhost')
)

channel = connection.channel()


method_frame, header_frame, body = channel.basic_get(
    queue="hello"
)

# method_frame, header_frame, body = channel.basic_get(
#     queue="hello",
#     auto_ack=True
# )

# rabbitmq won't know if msg consumed or not, so it will be available for consumption until we ack it
# if method_frame:
#     print(body.decode())

if method_frame:
    print(body.decode())

    channel.basic_ack(
        delivery_tag=method_frame.delivery_tag
    )


connection.close()

Hello RabbitMQ


---
## Chapter 2

In [5]:
'''
Exchange = Router
Queue = Storage

What is a Routing Key?
A Routing Key is simply a label attached to a message.

routing_key="order"
routing_key="payment"

Think of it as: Message + Address
The address is the Routing Key.

---

What is a Binding?
A Binding is a rule connecting:

Exchange
     |
     V
 Queue

Example:
Exchange ----order----> order_queue

This means:

If routing_key = order
send message to order_queue
'''

'\nExchange = Router\nQueue = Storage\n\nWhat is a Routing Key?\nA Routing Key is simply a label attached to a message.\n\nrouting_key="order"\nrouting_key="payment"\n\nThink of it as: Message + Address\nThe address is the Routing Key.\n\n---\n\nWhat is a Binding?\nA Binding is a rule connecting:\n\nExchange\n     |\n     V\n Queue\n\nExample:\nExchange ----order----> order_queue\n\nThis means:\n\nIf routing_key = order\nsend message to order_queue\n'

In [6]:
#### Let's say we have:
'''
Exchange: app_exchange

Queues:
- order_queue
- payment_queue

Bindings:

order_queue   -> order
payment_queue -> payment

Flow:

Producer
    |
    | routing_key=order
    V
app_exchange
    |
    V
order_queue
'''

'\nExchange: app_exchange\n\nQueues:\n- order_queue\n- payment_queue\n\nBindings:\n\norder_queue   -> order\npayment_queue -> payment\n\nFlow:\n\nProducer\n    |\n    | routing_key=order\n    V\napp_exchange\n    |\n    V\norder_queue\n'

In [7]:
'''
The Default Exchange

RabbitMQ automatically creates a built-in exchange.

Name:

exchange=""

Notice the empty string.

This is called the Default Exchange.

How Default Exchange Works

If:

routing_key="hello"

RabbitMQ looks for a queue named:

hello

and routes the message there.

Visual:

Producer
   |
   | routing_key=hello
   V
Default Exchange
   |
   V
hello Queue

That's why your first example worked.
'''

'\nThe Default Exchange\n\nRabbitMQ automatically creates a built-in exchange.\n\nName:\n\nexchange=""\n\nNotice the empty string.\n\nThis is called the Default Exchange.\n\nHow Default Exchange Works\n\nIf:\n\nrouting_key="hello"\n\nRabbitMQ looks for a queue named:\n\nhello\n\nand routes the message there.\n\nVisual:\n\nProducer\n   |\n   | routing_key=hello\n   V\nDefault Exchange\n   |\n   V\nhello Queue\n\nThat\'s why your first example worked.\n'

In [8]:
host = pika.ConnectionParameters('localhost')
connection = pika.BlockingConnection(host)
channel = connection.channel()

channel.queue_declare(
    queue="order_queue",
    durable=True
)

channel.queue_declare(
    queue="payment_queue",
    durable=True
)

<METHOD(['channel_number=1', 'frame_type=1', "method=<Queue.DeclareOk(['consumer_count=0', 'message_count=3', 'queue=payment_queue'])>"])>

In [9]:
channel.basic_publish(
    exchange='',
    routing_key='order_queue',
    body='order_created',
)

In [10]:
channel.basic_publish(
    exchange='',
    routing_key='payment_queue',
    body='payment_successful',
)

In [11]:
'''
Mental Model

By the end of Chapter 2, remember this:

Producer
    |
    V
Exchange
    |
    V
Queue
    |
    V
Consumer

and

Routing Key
     +
Binding
     =
Routing Decision
'''

'\nMental Model\n\nBy the end of Chapter 2, remember this:\n\nProducer\n    |\n    V\nExchange\n    |\n    V\nQueue\n    |\n    V\nConsumer\n\nand\n\nRouting Key\n     +\nBinding\n     =\nRouting Decision\n'

---

## Chapter - 3

In [12]:
'''
Direct Exchange

A Direct Exchange routes messages based on an exact routing key match.

Example:

Routing Key = order

goes to:

order_queue

Visual:

Producer
   |
   | routing_key=order
   V
Direct Exchange
   |
   V
order_queue
'''

'\nDirect Exchange\n\nA Direct Exchange routes messages based on an exact routing key match.\n\nExample:\n\nRouting Key = order\n\ngoes to:\n\norder_queue\n\nVisual:\n\nProducer\n   |\n   | routing_key=order\n   V\nDirect Exchange\n   |\n   V\norder_queue\n'

In [13]:
# Declare the Direct Exchange

channel.exchange_declare(
    exchange="app_exchange",
    exchange_type="direct",
    durable=True
)

<METHOD(['channel_number=1', 'frame_type=1', 'method=<Exchange.DeclareOk>'])>

In [14]:
# Declare the Queues

channel.queue_declare(
    queue="order_queue",
    durable=True
)

channel.queue_declare(
    queue="payment_queue",
    durable=True
)

<METHOD(['channel_number=1', 'frame_type=1', "method=<Queue.DeclareOk(['consumer_count=0', 'message_count=4', 'queue=payment_queue'])>"])>

In [15]:
# Bind the Queues to the Exchange with Routing Keys

channel.queue_bind(
    exchange="app_exchange",
    queue="order_queue",
    routing_key="order"
)

<METHOD(['channel_number=1', 'frame_type=1', 'method=<Queue.BindOk>'])>

In [16]:
# Bind the Queues to the Exchange with Routing Keys

channel.queue_bind(
    exchange="app_exchange",
    queue="payment_queue",
    routing_key="payment"
)

<METHOD(['channel_number=1', 'frame_type=1', 'method=<Queue.BindOk>'])>

In [17]:
# Publish Messages with Routing Keys

channel.basic_publish(
    exchange="app_exchange",
    routing_key="order",
    body="Order #101 Created"
)